# Emotion-Conditioned IMTalker Live Pipeline — RunPod Deployment

This notebook launches the **emotion-aware PersonaPlex + IMTalker live server** on RunPod.

It extends the base `RunPod_RTX5090_PersonaPlex_IMTalker_Live_fixed.ipynb` with:
- **Autonomously downloading** your trained emotion LoRA adapter from Hugging Face:
  [`bitwisemind/emotion_lora_adapter_v1`](https://huggingface.co/bitwisemind/emotion_lora_adapter_v1) (`last.ckpt`)
- Passing the desired **emotion ID** (neutral, happy, sad, angry, surprise, fear) to the avatar
- Real-time conversational speech-to-face pipeline with high lip-sync accuracy and emotional facial expressions
- Fully resilient asset downloads (supporting both HF `dataset` and `model` repositories, guarded fast-transfer)
- Modern non-deprecated Hugging Face CLI tooling (`hf download`, `hf auth whoami`)

**Pipeline:**
```
PersonaPlex audio + hidden states (12.5 Hz)
  -> RMS gate -> 0.96s lookahead adapter
  -> IMTalker generator (base + style LoRA + emotion LoRA)
  -> IMTalker FP32 renderer
  -> Opus audio (/ws/conversation) + JPEG video (/ws/video) websockets
```

## How to use

1. **Select your desired emotion** in the PARAMETERS cell below (`EMOTION = "happy"`, `"sad"`, `"angry"`, etc.)
2. Run cells top-to-bottom
3. The server will download all checkpoints (including your emotion LoRA from Hugging Face) and launch the live web service


In [ ]:
# If the repo is not yet cloned, run this cell first
# (comment out if you already have the checkout)
!git clone https://github.com/MoshiHead/n_IMT_s_Perso_v2_2_aug_compress_log_Emotion_anti_v1.git speech2avatar


## Parameters

Edit the section below before running. Emotion-specific settings are in the "Emotion" block.

In [ ]:
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# Project location
# ---------------------------------------------------------------------------
PROJECT_ROOT = "/workspace/speech2avatar"
if not Path(PROJECT_ROOT).exists() and Path("/workspace/n_IMTalker_s_Personaplex_v2_2_aug_demo_code_add_time_nd_compress_log").exists():
    PROJECT_ROOT = "/workspace/n_IMTalker_s_Personaplex_v2_2_aug_demo_code_add_time_nd_compress_log"
elif not Path(PROJECT_ROOT).exists() and Path("/workspace/IMTalker").exists():
    PROJECT_ROOT = "/workspace"

GIT_REPO_URL = "https://github.com/MoshiHead/n_IMT_s_Perso_v2_2_aug_compress_log_Emotion_anti_v1.git"
GIT_BRANCH   = "main"

# ---------------------------------------------------------------------------
# Toolchain
# ---------------------------------------------------------------------------
VENV_DIR = "/workspace/preprocess_5090"

# ---------------------------------------------------------------------------
# Service
# ---------------------------------------------------------------------------
HOST = "0.0.0.0"
PORT = 8998
RUNPOD_POD_ID = os.environ.get("RUNPOD_POD_ID", "")
PUBLIC_URL = f"https://{RUNPOD_POD_ID}-{PORT}.proxy.runpod.net/" if RUNPOD_POD_ID else None

# ---------------------------------------------------------------------------
# EMOTION — edit this block to change the avatar's expression
# ---------------------------------------------------------------------------
# Valid values: "neutral", "happy", "sad", "angry", "surprise", "fear"
EMOTION = "happy"

EMOTION_MAP = {"neutral": 0, "happy": 1, "sad": 2, "angry": 3, "surprise": 4, "fear": 5}
EMOTION_ID  = EMOTION_MAP.get(EMOTION.lower(), 1)

# Trained Emotion LoRA from Hugging Face
HF_EMOTION_LORA_REPO = "bitwisemind/emotion_lora_adapter_v1"
HF_EMOTION_LORA_FILE = "last.ckpt"

# ---------------------------------------------------------------------------
# Hugging Face sources (base live assets)
# Note: niloy629/hdtf_preprocess is a dataset repo (repo_type="dataset")
# ---------------------------------------------------------------------------
HF_ADAPTER_REPO = "niloy629/hdtf_preprocess"
HF_ADAPTER_FILE = "personaplex_lookahead_rms_adapter/checkpoints/personaplex_lookahead096_future048_rms50_adapter.pt"
HF_SILENCE_FILE = "personaplex_lookahead_rms_adapter/stats/silence_helium_mean.pt"

HF_LORA_REPO   = "niloy629/hdtf_preprocess"
HF_LORA_FILE   = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
HF_LORA_SHA256 = "f34de1bd748857bc802102da578046bb10bd5f664460607d9c785ff35922e52f"

HF_BLINK_REPO   = "niloy629/hdtf_preprocess"
HF_BLINK_FILE   = "lora/3robert_audio3_ditto_static_motion.pt"
HF_BLINK_SHA256 = "e29a41ff004b228d7efee15cad0f32f4d4bc5466563709e2ba78b158d4e340bb"

HF_IMTALKER_REPO  = "cbsjtu01/IMTalker"
HF_IMTALKER_FILES = [
    "config.yaml",
    "renderer.ckpt",
    "generator.ckpt",
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/feature_extractor_config.json",
]

HF_PERSONAPLEX_REPO = "brianmatzelle/personaplex-7b-v1-bnb-4bit"
HF_PERSONAPLEX_MIMI_REPO = "nvidia/personaplex-7b-v1"
HF_PERSONAPLEX_MIMI_FILES = [
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "tokenizer_spm_32k_3.model",
]
HF_VOICES_FILE = "voices.tgz"

# ---------------------------------------------------------------------------
# Local asset destinations (derived from PROJECT_ROOT)
# ---------------------------------------------------------------------------
CHECKPOINT_DIR    = os.path.join(PROJECT_ROOT, "checkpoints")
IMTALKER_DIR      = os.path.join(PROJECT_ROOT, "IMTalker")
IMTALKER_CKPT_DIR = os.path.join(IMTALKER_DIR, "checkpoints")
PERSONAPLEX_DIR   = os.path.join(CHECKPOINT_DIR, "personaplex_bnb4")
HF_ASSETS_DIR     = "/workspace/hf_assets"
VENV_PYTHON       = os.path.join(VENV_DIR, "bin", "python")
VENV_ACTIVATE     = f"source {VENV_DIR}/bin/activate"
EXPECTED_TORCH_VERSION = "2.8.0+cu128"

EMOTION_LORA_DIR  = os.path.join(CHECKPOINT_DIR, "emotion_lora")
EMOTION_LORA_CKPT = os.path.join(EMOTION_LORA_DIR, HF_EMOTION_LORA_FILE)
EMOTION_LORA_RANK = 32     # matches training LORA_RANK
EMOTION_LORA_ALPHA= 64.0   # matches training LORA_ALPHA

# Derived paths (matching download_live_assets.sh / start_winner_live.sh)
ADAPTER_PATH        = os.path.join(CHECKPOINT_DIR, HF_ADAPTER_FILE)
SILENCE_HELIUM_PATH = os.path.join(CHECKPOINT_DIR, HF_SILENCE_FILE)
LORA_CKPT_LOCAL     = os.path.join(CHECKPOINT_DIR, HF_LORA_FILE)
BLINK_CKPT_LOCAL    = os.path.join(CHECKPOINT_DIR, HF_BLINK_FILE)
GENERATOR_CKPT      = os.path.join(IMTALKER_CKPT_DIR, "generator.ckpt")
RENDERER_CKPT       = os.path.join(IMTALKER_CKPT_DIR, "renderer.ckpt")
WAV2VEC_DIR         = os.path.join(IMTALKER_CKPT_DIR, "wav2vec2-base-960h")
PERSONAPLEX_WEIGHT_PATH = os.path.join(PERSONAPLEX_DIR, "model_bnb_4bit.pt")
VOICE_PROMPT_PATH   = os.path.join(PERSONAPLEX_DIR, "voices", "VARM3.pt")

LOG_PATH = os.path.join(PROJECT_ROOT, f"live_emotion_{EMOTION}_p{PORT}.log")
PID_PATH = os.path.join(PROJECT_ROOT, f"live_emotion_{EMOTION}_p{PORT}.pid")

print(f"Emotion:           {EMOTION} (id={EMOTION_ID})")
print(f"Emotion LoRA repo: {HF_EMOTION_LORA_REPO}")
print(f"Emotion LoRA ckpt: {EMOTION_LORA_CKPT}")
print(f"Port:              {PORT}  {'-> ' + PUBLIC_URL if PUBLIC_URL else '(no RunPod proxy URL detected)'}")
print(f"Project root:      {PROJECT_ROOT}")
print(f"Venv:              {VENV_DIR}")


## Utilities

Shared helpers (streaming subprocess runner, torch probe, sha256 checker, HF downloader with dataset support, port helpers).

In [ ]:
import hashlib
import json as _json
import socket
import subprocess
import time
from pathlib import Path


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False):
    last_returncode = None
    env_copy = (env.copy() if env else os.environ.copy())
    if "PATH" in env_copy:
        env_copy["PATH"] = f"{VENV_DIR}/bin:{env_copy['PATH']}"
    else:
        env_copy["PATH"] = f"{VENV_DIR}/bin"

    for attempt in range(1, retries + 1):
        if not quiet:
            attempt_str = f"   [attempt {attempt}/{retries}]" if retries > 1 else ""
            print(f"$ {cmd}{attempt_str}")
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=cwd,
            env=env_copy,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            executable="/bin/bash",
        )
        if proc.stdout:
            for line in proc.stdout:
                if not quiet:
                    print(line, end="")
        proc.wait()
        last_returncode = proc.returncode
        if proc.returncode == 0:
            return 0
        if attempt < retries:
            if not quiet:
                print(f"[warn] command failed with exit code {proc.returncode}")
                print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)

    if check:
        raise RuntimeError(
            f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}"
        )
    return last_returncode


def get_torch_info():
    probe = "\n".join([
        "import torch, json",
        "try:",
        "    res = {",
        "        'version': torch.__version__,",
        "        'cuda_available': torch.cuda.is_available(),",
        "        'cuda_version': torch.version.cuda,",
        "        'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,",
        "    }",
        "except Exception as e:",
        "    res = {'error': str(e)}",
        "print('TORCH_JSON_START' + json.dumps(res))",
    ])
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout); print(out.stderr)
        raise RuntimeError(f"Failed to query torch/CUDA state inside the venv: {out.stderr}")
    lines = [l for l in out.stdout.splitlines() if "TORCH_JSON_START" in l]
    if not lines:
        raise RuntimeError(f"No torch info output found: {out.stdout}")
    payload = _json.loads(lines[-1].replace("TORCH_JSON_START", ""))
    if "error" in payload:
        raise RuntimeError(f"Torch import error: {payload['error']}")
    return payload


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def hf_download(repo, filename, local_dir, repo_type="model", token=None, retries=3, quiet=False):
    tok = token or globals().get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
    dest = Path(local_dir) / filename
    if dest.exists() and dest.stat().st_size > 0:
        if not quiet:
            print(f"[skip] already exists: {dest}")
        return str(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    tok_env = f"HF_TOKEN={tok} " if tok else ""
    repo_type_flag = f"--repo-type {repo_type}" if repo_type != "model" else ""

    # Guard: only enable HF_HUB_ENABLE_HF_TRANSFER if hf_transfer is actually installed in the venv
    # This prevents 'ValueError: Fast download using hf_transfer is enabled but hf_transfer package is not available'
    hf_transfer_guard = (
        "python -c 'import hf_transfer' >/dev/null 2>&1 && "
        "export HF_HUB_ENABLE_HF_TRANSFER=1 || export HF_HUB_ENABLE_HF_TRANSFER=0"
    )

    # Use modern non-deprecated 'hf download' (with fallback to huggingface-cli if needed)
    cmd = (
        f"{VENV_ACTIVATE} && {hf_transfer_guard} && "
        f"({tok_env}hf download {repo} {filename} {repo_type_flag} --local-dir {local_dir} || "
        f"{tok_env}huggingface-cli download {repo} {filename} {repo_type_flag} --local-dir {local_dir})"
    )
    run(cmd, retries=retries, retry_delay=10, quiet=quiet)
    return str(dest)


def verify_checksum_with_recovery(path, expected_sha256, repo, filename, repo_type, local_dir, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        if not os.path.exists(path):
            print(f"[fix] missing file, downloading: {path} (attempt {attempt}/{max_attempts})")
            hf_download(repo, filename, local_dir, repo_type=repo_type)
        actual = sha256_of(path)
        if actual.lower() == expected_sha256.lower():
            print(f"[ok] checksum verified: {path}")
            return True
        print(f"[warn] checksum mismatch for {path}")
        print(f"       expected: {expected_sha256}")
        print(f"       actual:   {actual}")
        try:
            os.remove(path)
        except FileNotFoundError:
            pass
        hf_download(repo, filename, local_dir, repo_type=repo_type)
    raise RuntimeError(f"Checksum verification failed after {max_attempts} attempts: {path}")


def port_listening(host, port, timeout=0.5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def find_pids_on_port(port):
    result = subprocess.run(
        f"lsof -ti tcp:{port} 2>/dev/null || true",
        shell=True, capture_output=True, text=True,
    )
    return [p.strip() for p in result.stdout.strip().splitlines() if p.strip()]


def tail_log(n=100):
    if not os.path.exists(LOG_PATH):
        print(f"[info] no log at {LOG_PATH} yet")
        return
    run(f"tail -n {n} '{LOG_PATH}'", check=False)


print("[utilities] OK")


## 1 — Clone / validate repository

In [ ]:
from pathlib import Path

imtalker_server = Path(IMTALKER_DIR) / "liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py"

if Path(PROJECT_ROOT).exists() and imtalker_server.exists():
    print(f"[ok] repo already present at {PROJECT_ROOT}")
    run(f"find '{PROJECT_ROOT}' -name '*.sh' -exec chmod +x {{}} \\;", check=False, quiet=True)
else:
    if GIT_REPO_URL:
        print(f"Cloning {GIT_REPO_URL} -> {PROJECT_ROOT}")
        run(f"git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {PROJECT_ROOT}", retries=2)
    else:
        raise RuntimeError(f"Repo not found at {PROJECT_ROOT} and GIT_REPO_URL is not set")

# Validate key files
required_files = [
    imtalker_server,
    Path(IMTALKER_DIR) / "start_winner_live.sh",
    Path(IMTALKER_DIR) / "start_typeah_audio_pace_varm3.sh",
    Path(PROJECT_ROOT) / "run_live.sh",
]
for f in required_files:
    status = "[ok]" if f.exists() else "[MISSING]"
    print(f"  {status} {f}")


## 2 — Install system packages and Python venv

> Prepares python venv with torch, typing_extensions, hf_transfer, and IMTalker dependencies.

In [ ]:
# 1. Install system packages
run("apt-get update -qq && apt-get install -y -qq libsndfile1 libgl1 ffmpeg", check=False, retries=2)

# 2. Create venv if missing
if not Path(VENV_PYTHON).exists():
    run(f"python3.11 -m venv {VENV_DIR}", retries=2)
    print(f"[ok] venv created at {VENV_DIR}")

# 3. Upgrade pip, setuptools, wheel
run(
    f"{VENV_ACTIVATE} && pip install --quiet --upgrade pip setuptools wheel",
    retries=2,
)

# 4. Install huggingface_hub[cli] and hf_transfer into the venv (matching reference notebook)
run(
    f'{VENV_ACTIVATE} && pip install --quiet "huggingface_hub[cli]" hf_transfer typing_extensions filelock jinja2 fsspec networkx sympy pyyaml',
    retries=2,
)

# 5. Also install into current notebook Python environment so kernel imports work
import sys
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub[cli]", "hf_transfer"], check=False)
except Exception:
    pass

# 6. Check / install torch pin
info = None
try:
    info = get_torch_info()
    print(f"[info] torch={info['version']}")
except Exception:
    info = None

if info is None or info.get("version") != EXPECTED_TORCH_VERSION:
    print(f"[install] installing pinned torch {EXPECTED_TORCH_VERSION}...")
    run(
        f"{VENV_ACTIVATE} && pip install --quiet --no-deps "
        "torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
        "--index-url https://download.pytorch.org/whl/cu128",
        retries=2,
    )

# 7. Install IMTalker dependencies (handle both requirement.txt and requirements.txt)
req_candidates = [
    Path(IMTALKER_DIR) / "requirement.txt",
    Path(IMTALKER_DIR) / "requirements.txt",
]
req_file = next((f for f in req_candidates if f.exists()), None)
if req_file:
    print(f"[install] installing IMTalker dependencies from {req_file.name}...")
    run(f"{VENV_ACTIVATE} && pip install --quiet -r {req_file}", retries=2, check=False)
else:
    print("[warn] requirement.txt not found in IMTalker, installing core packages directly...")
    run(
        f"{VENV_ACTIVATE} && pip install --quiet 'numpy<2.0.0' 'pandas<2.2.0' av==12.0.0 "
        "face_alignment==1.4.1 pytorch-lightning==2.2.1 torchdiffeq==0.2.5 timm==1.0.9 "
        "transformers librosa tensorboard gradio fastapi pydantic",
        retries=2, check=False,
    )

# 8. Install PersonaPlex package
if (Path(PERSONAPLEX_DIR) / "moshi" / "setup.py").exists():
    run(f"{VENV_ACTIVATE} && pip install --quiet --no-deps -e {PERSONAPLEX_DIR}/moshi", check=False)

print("\n[ok] environment ready")
info = get_torch_info()
print(f"[ok] torch={info['version']} | cuda={info['cuda_version']} | gpu={info['device_name']}")


## 3 — Hugging Face authentication

You need a HF token with access to `nvidia/personaplex-7b-v1` (accept the model card on HF first).

In [ ]:
import os, getpass, subprocess, json

HF_TOKEN = os.environ.get("HF_TOKEN", "_GjsPkoyTVNNHDFJNYQCBGQGdYUdKgadHsI")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face token (input hidden): ")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# 1. Pure Python verification via venv Python (zero CLI deprecation warnings)
verified = False
probe_script = "\n".join([
    "import json",
    "from huggingface_hub import HfApi",
    f"api = HfApi(token='{HF_TOKEN}')",
    "try:",
    "    info = api.whoami()",
    "    username = info.get('name') or info.get('fullname') or 'authenticated user'",
    "    print('AUTH_RESULT:' + json.dumps({'success': True, 'user': username}))",
    "except Exception as e:",
    "    print('AUTH_RESULT:' + json.dumps({'success': False, 'error': str(e)}))",
])

try:
    proc = subprocess.run([VENV_PYTHON, "-c", probe_script], capture_output=True, text=True)
    for line in proc.stdout.splitlines():
        if line.startswith("AUTH_RESULT:"):
            res = json.loads(line.replace("AUTH_RESULT:", "", 1))
            if res.get("success"):
                print(f"[ok] Hugging Face authentication ready for: {res['user']}")
                verified = True
            else:
                print(f"[warn] HF authentication returned: {res.get('error')}")
            break
except Exception as exc:
    print(f"[warn] Could not run HF probe in venv: {exc}")

# 2. Modern CLI fallback if needed ('hf auth whoami', not deprecated 'huggingface-cli whoami')
if not verified:
    print("[info] trying CLI verification via 'hf auth whoami'...")
    code = run(f"{VENV_ACTIVATE} && HF_TOKEN={HF_TOKEN} hf auth whoami", check=False, retries=1)
    if code == 0:
        verified = True

if verified:
    print("[ok] HF authentication ready")
else:
    print("[warn] Proceeding with provided token...")


## 4 — Download runtime assets

Downloads adapter, silence seed, style LoRA, blink motion, IMTalker checkpoints, PersonaPlex weights, and your newly trained **emotion LoRA adapter** from Hugging Face.

> Correctly handles both Hugging Face `dataset` repositories (`niloy629/hdtf_preprocess`) and `model` repositories (`bitwisemind/emotion_lora_adapter_v1`, `cbsjtu01/IMTalker`). Fast transfer via `hf_transfer` is used if available, without risk of crashes. Already-downloaded files are skipped automatically.

In [ ]:
import tarfile
import shutil
from pathlib import Path

# --- 1. IMTalker pretrained models (cbsjtu01/IMTalker, model repo) ---
print("[1/7] IMTalker pretrained checkpoints...")
for hf_file in HF_IMTALKER_FILES:
    dest = Path(IMTALKER_CKPT_DIR) / hf_file
    if not dest.exists():
        hf_download(HF_IMTALKER_REPO, hf_file, IMTALKER_CKPT_DIR, repo_type="model")
    else:
        print(f"[skip] {dest.name}")

# --- 2. Lookahead / RMS adapter + silence seed (niloy629/hdtf_preprocess, dataset repo) ---
print("\n[2/7] PersonaPlex lookahead/RMS adapter and silence seed...")
hf_download(HF_ADAPTER_REPO, HF_ADAPTER_FILE, CHECKPOINT_DIR, repo_type="dataset")
hf_download(HF_ADAPTER_REPO, HF_SILENCE_FILE, CHECKPOINT_DIR, repo_type="dataset")

# --- 3. Style LoRA & blink motion map with checksum verification (niloy629/hdtf_preprocess, dataset repo) ---
print("\n[3/7] Style LoRA & blink motion map...")
verify_checksum_with_recovery(
    LORA_CKPT_LOCAL, HF_LORA_SHA256, HF_LORA_REPO, HF_LORA_FILE, "dataset", CHECKPOINT_DIR,
)
verify_checksum_with_recovery(
    BLINK_CKPT_LOCAL, HF_BLINK_SHA256, HF_BLINK_REPO, HF_BLINK_FILE, "dataset", CHECKPOINT_DIR,
)

# --- 4. PersonaPlex 4-bit weights & Mimi tokenizer (brianmatzelle + nvidia, model repos) ---
print("\n[4/7] PersonaPlex weights & tokenizer...")
os.makedirs(PERSONAPLEX_DIR, exist_ok=True)
hf_download(HF_PERSONAPLEX_REPO, "model_bnb_4bit.pt", PERSONAPLEX_DIR, repo_type="model")
for fname in HF_PERSONAPLEX_MIMI_FILES:
    hf_download(HF_PERSONAPLEX_MIMI_REPO, fname, PERSONAPLEX_DIR, repo_type="model")

# --- 5. PersonaPlex voice prompt (voices.tgz from nvidia/personaplex-7b-v1) ---
print("\n[5/7] PersonaPlex voice prompt (VARM3.pt)...")
if not os.path.exists(VOICE_PROMPT_PATH):
    print(f"[download] downloading and extracting {HF_VOICES_FILE}...")
    hf_download(HF_PERSONAPLEX_MIMI_REPO, HF_VOICES_FILE, PERSONAPLEX_DIR, repo_type="model")
    voices_tgz = os.path.join(PERSONAPLEX_DIR, HF_VOICES_FILE)
    with tarfile.open(voices_tgz) as tf:
        tf.extractall(PERSONAPLEX_DIR)
else:
    print(f"[skip] voice prompt exists: {VOICE_PROMPT_PATH}")

# --- 6. Emotion LoRA adapter (bitwisemind/emotion_lora_adapter_v1, model repo) ---
print("\n[6/7] Emotion LoRA adapter (bitwisemind/emotion_lora_adapter_v1)...")
emo_dest = Path(EMOTION_LORA_CKPT)
if not emo_dest.exists() or emo_dest.stat().st_size == 0:
    local_exp_ckpt = Path("/workspace/emotion_exps/emotion_lora_v1/checkpoints/last.ckpt")
    if local_exp_ckpt.exists():
        print(f"[found] copying local trained checkpoint: {local_exp_ckpt} -> {emo_dest}")
        emo_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(local_exp_ckpt), str(emo_dest))
    else:
        print(f"[download] downloading trained emotion LoRA from {HF_EMOTION_LORA_REPO}/{HF_EMOTION_LORA_FILE} ...")
        hf_download(HF_EMOTION_LORA_REPO, HF_EMOTION_LORA_FILE, EMOTION_LORA_DIR, repo_type="model")
        print(f"  ✓ Emotion LoRA downloaded: {emo_dest}")
else:
    print(f"[skip] Emotion LoRA already exists: {emo_dest} ({emo_dest.stat().st_size / (1 << 20):.1f} MB)")

# --- 7. Setup directory compatibility symlinks ---
os.makedirs(HF_ASSETS_DIR, exist_ok=True)
for sub in ["personaplex_lookahead_rms_adapter", "lora", "live_winner"]:
    src_sub = os.path.join(CHECKPOINT_DIR, sub)
    dst_sub = os.path.join(HF_ASSETS_DIR, sub)
    if os.path.exists(src_sub) and not os.path.exists(dst_sub):
        try:
            os.symlink(src_sub, dst_sub)
        except Exception:
            pass

if not os.path.exists("/workspace/personaplex_bnb4") and os.path.exists(PERSONAPLEX_DIR):
    try:
        os.symlink(PERSONAPLEX_DIR, "/workspace/personaplex_bnb4")
    except Exception:
        pass

print("\n" + "="*50)
print("Runtime asset verification:")
all_assets = [
    ("IMTalker generator", GENERATOR_CKPT),
    ("IMTalker renderer", RENDERER_CKPT),
    ("Lookahead adapter", ADAPTER_PATH),
    ("Silence helium seed", SILENCE_HELIUM_PATH),
    ("Live winner style LoRA", LORA_CKPT_LOCAL),
    ("Blink motion", BLINK_CKPT_LOCAL),
    ("PersonaPlex model", PERSONAPLEX_WEIGHT_PATH),
    ("Voice prompt (VARM3)", VOICE_PROMPT_PATH),
    ("Emotion LoRA adapter", EMOTION_LORA_CKPT),
]
for label, p in all_assets:
    exists = os.path.exists(p)
    sz = f"{os.path.getsize(p)/(1<<20):.1f} MB" if exists else "MISSING"
    status = "[OK]" if exists else "[MISSING]"
    print(f"  {status:9s} {label:25s}: {p} ({sz})")

missing = [lbl for lbl, p in all_assets if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"Assets still missing: {missing}")

print("\n[ok] all runtime assets downloaded and verified!")


## 5 — Validate emotion LoRA checkpoint

Verifies that the emotion LoRA checkpoint exists and contains the expected emotion conditioning keys.

In [ ]:
import sys
from pathlib import Path

# If emotion LoRA is not set, print an info message and continue with neutral
if not EMOTION_LORA_CKPT or not Path(EMOTION_LORA_CKPT).exists():
    if EMOTION_LORA_CKPT:
        print(f"[WARN] Emotion LoRA checkpoint not found: {EMOTION_LORA_CKPT}")
        print("       Running with base emotion_conditioner only (zero-init = neutral behaviour)")
    else:
        print("[info] EMOTION_LORA_CKPT not set — using base emotion_conditioner (neutral)")
    _HAS_EMOTION_LORA = False
else:
    # Validate checkpoint contents using venv Python to avoid kernel dep issues
    check_env = os.environ.copy()
    check_env["CHECK_EMO_CKPT"] = str(EMOTION_LORA_CKPT)
    check_script = "\n".join([
        "import torch, json, os",
        "ckpt_path = os.environ.get('CHECK_EMO_CKPT', '')",
        "ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)",
        "state = ckpt.get('ema_state_dict') or ckpt.get('state_dict', ckpt)",
        "if isinstance(state, dict) and 'model' in state and isinstance(state['model'], dict):",
        "    state = state['model']",
        "state = {k.replace('model.', '', 1) if k.startswith('model.') else k: v for k, v in state.items()}",
        "emotion_keys = [k for k in state if 'emotion' in k]",
        "lora_keys    = [k for k in state if 'lora_'  in k]",
        "proj_w = state.get('emotion_conditioner.emotion_projection.0.weight')",
        "norm = proj_w.norm().item() if proj_w is not None else None",
        "print(json.dumps({'emotion_keys': len(emotion_keys), 'lora_keys': len(lora_keys), 'proj_norm': norm}))",
    ])
    result = subprocess.run(
        [VENV_PYTHON, "-c", check_script],
        env=check_env,
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print("[WARN] Could not verify emotion LoRA checkpoint in venv:")
        print(result.stderr[:500])
        _HAS_EMOTION_LORA = True
    else:
        import json as _j
        info = _j.loads(result.stdout.strip().splitlines()[-1])
        print(f"[Emotion LoRA] emotion_keys={info['emotion_keys']}  lora_keys={info['lora_keys']}")
        if info.get("proj_norm") is not None:
            norm = info["proj_norm"]
            if norm < 1e-6:
                print("  [WARN] emotion_projection weights are still zero")
            else:
                print(f"  [ok] emotion_projection weight norm={norm:.6f}")
        if info["emotion_keys"] == 0:
            print("  [WARN] no emotion_conditioner keys found — may be wrong checkpoint")
        _HAS_EMOTION_LORA = True

print(f"\nActive emotion  : {EMOTION} (id={EMOTION_ID})")
print(f"Has emotion LoRA: {_HAS_EMOTION_LORA}")


## 6 — GPU / CUDA validation

In [ ]:
run("nvidia-smi", check=False)
info = get_torch_info()
print(f"\n[ok] torch={info['version']} | cuda={info['cuda_version']} | gpu={info['device_name']}")
assert info["cuda_available"], "CUDA not available — check GPU selection in RunPod"


## 7 — Pre-flight asset checks

In [ ]:
required_assets = [
    ("generator.ckpt",          GENERATOR_CKPT),
    ("renderer.ckpt",           RENDERER_CKPT),
    ("wav2vec2-base-960h/",     WAV2VEC_DIR),
    ("adapter.pt",              ADAPTER_PATH),
    ("silence_helium.pt",       SILENCE_HELIUM_PATH),
    ("style_lora.ckpt",         LORA_CKPT_LOCAL),
    ("blink_motion.pt",         BLINK_CKPT_LOCAL),
    ("PersonaPlex model_bnb4",  PERSONAPLEX_WEIGHT_PATH),
    ("PersonaPlex voice VARM3", VOICE_PROMPT_PATH),
]
if _HAS_EMOTION_LORA:
    required_assets.append(("emotion_lora.ckpt", EMOTION_LORA_CKPT))

all_ok = True
for label, path in required_assets:
    exists = Path(path).exists()
    status = "[ok]" if exists else "[MISSING]"
    print(f"  {status} {label}: {path}")
    if not exists:
        all_ok = False

if not all_ok:
    raise RuntimeError("One or more required assets are missing. See [MISSING] entries above.")

fm_py = Path(IMTALKER_DIR) / "generator" / "FM.py"
live_py = Path(IMTALKER_DIR) / "liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py"
if "EmotionConditioner" not in fm_py.read_text():
    print("[WARN] EmotionConditioner not in FM.py — emotion conditioning may not work")
else:
    print("  [ok] EmotionConditioner in FM.py")
if "emotion_id" not in live_py.read_text():
    print("[WARN] emotion_id not in live server — check that start_winner_live.sh and the server were patched")
else:
    print("  [ok] emotion_id injection in live server")


## 8 — Launch the emotion-conditioned live server

Sets the emotion environment variables and calls `run_live.sh` (exactly like the base live notebook — never reimplementing the start flags).

In [ ]:
# Free any existing server on this port
existing_pids = find_pids_on_port(PORT)
if existing_pids:
    print(f"[warn] port {PORT} in use by pids {existing_pids}; killing...")
    for pid in existing_pids:
        subprocess.run(["kill", "-9", pid], check=False)
    time.sleep(1)

# Build env for the launch
launch_env = os.environ.copy()
launch_env.update({
    "PORT":                str(PORT),
    "SPEECH2AVATAR_ROOT":  PROJECT_ROOT,
    "IMTALKER_DIR":        IMTALKER_DIR,
    "VENV_DIR":            VENV_DIR,
    "LORA_GENERATOR_PATH": LORA_CKPT_LOCAL,
    # Emotion vars (read by start_winner_live.sh -> live server argparse)
    "EMOTION_ID":          str(EMOTION_ID),
    "EMOTION_LORA_PATH":   EMOTION_LORA_CKPT if _HAS_EMOTION_LORA else "",
    "EMOTION_LORA_RANK":   str(EMOTION_LORA_RANK),
    "EMOTION_LORA_ALPHA":  str(EMOTION_LORA_ALPHA),
})

print(f"Launching emotion-aware live server with EMOTION={EMOTION} (id={EMOTION_ID})")
print(f"Emotion LoRA ckpt: {EMOTION_LORA_CKPT if _HAS_EMOTION_LORA else '(none)'}")
print(f"Log: {LOG_PATH}")
print(f"PID: {PID_PATH}")

run_live_sh = os.path.join(PROJECT_ROOT, "run_live.sh")
with open(LOG_PATH, "w") as logf:
    proc = subprocess.Popen(
        ["bash", run_live_sh],
        env=launch_env,
        cwd=PROJECT_ROOT,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

Path(PID_PATH).write_text(str(proc.pid))
print(f"[ok] server started | pid={proc.pid} | port={PORT}")
if PUBLIC_URL:
    print(f"[url] {PUBLIC_URL}")

# Wait for healthy startup markers
print("\nWaiting for server to be ready...", end="", flush=True)
ready_markers = ["[liveTryHeliumFM] server ready", "Application startup complete"]
for _ in range(120):  # up to 120s
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        log_text = Path(LOG_PATH).read_text(errors="ignore")
        if any(m in log_text for m in ready_markers):
            print("\n[ok] server is READY")
            break
        if proc.poll() is not None:
            print("\n[ERROR] server process exited prematurely! Check the log:")
            run(f"tail -n 50 '{LOG_PATH}'", check=False)
            raise RuntimeError("Server exited before becoming ready")
    except FileNotFoundError:
        pass
else:
    print("\n[warn] timed out waiting for ready marker — check log for errors")
    run(f"tail -n 30 '{LOG_PATH}'", check=False)

print(f"\nPort listening: {port_listening('127.0.0.1', PORT)}")


## Monitor

Re-run this cell at any time to see recent log output.

In [ ]:
tail_log(120)


## Switch Emotion

Emotion is set once at server startup (weights are loaded into GPU memory). To switch expressions, stop the server and relaunch with a different `EMOTION` value.

In [ ]:
# Instructions for switching emotion
NEW_EMOTION = "sad"  # change to: "neutral", "happy", "sad", "angry", "surprise", "fear"
NEW_EMOTION_ID = EMOTION_MAP.get(NEW_EMOTION)

print(f"Currently running: EMOTION={EMOTION} (id={EMOTION_ID})")
print()
print(f"To switch to '{NEW_EMOTION}' (id={NEW_EMOTION_ID}):")
print("  1. Run the STOP SERVER cell below")
print(f"  2. In the PARAMETERS cell, set:  EMOTION = \"{NEW_EMOTION}\"")
print("  3. Re-run the PARAMETERS cell and the LAUNCH cell")
print()
print("The emotion LoRA is baked into the GPU weights at startup — live switching")
print("without a restart is not supported in this version.")


## Full Diagnostics

Safe to run at any time, including during a failure.

In [ ]:
def run_full_diagnostics():
    print("== GPU ==")
    run("nvidia-smi", check=False)

    print("\n== Torch / CUDA (venv) ==")
    try:
        print(get_torch_info())
    except Exception as e:
        print(f"[error] torch check failed: {e}")

    print(f"\n== Port {PORT} ==")
    print(f"listening: {port_listening('127.0.0.1', PORT)}  pids: {find_pids_on_port(PORT)}")

    print(f"\n== Emotion ==")
    print(f"  EMOTION={EMOTION} (id={EMOTION_ID})")
    print(f"  emotion_lora_path={EMOTION_LORA_CKPT or '(none)'}")
    print(f"  has_emotion_lora={_HAS_EMOTION_LORA}")

    print("\n== Log tail ==")
    tail_log(80)


run_full_diagnostics()


## Stop Server

Set `STOP_SERVER = True` and re-run this cell.

In [ ]:
STOP_SERVER = False


def stop_server():
    if not os.path.exists(PID_PATH):
        print("[info] no pid file found; nothing to stop")
        return
    pid = open(PID_PATH).read().strip()
    print(f"[stop] terminating live server pid {pid}")
    subprocess.run(["kill", "-9", pid], check=False)
    try:
        os.remove(PID_PATH)
    except FileNotFoundError:
        pass
    print("[ok] server stopped")


if STOP_SERVER:
    stop_server()
else:
    print("STOP_SERVER is False — set it to True and re-run this cell to stop the server.")
